# NB_01 — Absorber Manufacturing Synthesis

This notebook uses the completed engineering source records:

```text
SOURCE_00_becker_transition_models.yaml
SOURCE_01_bismuth_microstructure.yaml
SOURCE_02_eliminating_nongaussian_spectral_response.yaml
```

It does **not** re-read the papers. It treats the source records as the engineering evidence layer and asks:

- Which variables recur across sources?
- Which relationships are supported by more than one source?
- Which quantitative values can be compared now?
- Which manufacturing specifications are supported?
- Which specifications remain unresolved?
- What should the next engineering notebook measure or model?

Outputs are written to:

```text
outputs/engineering_questions/absorber_manufacturing/SYNTHESIS_01/
```

and packaged as:

```text
exports/SYNTHESIS_01_export.zip
```

Run from top to bottom.


## 1. Configuration and repository paths

In [3]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

import pandas as pd
import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE: str | Path | None = None

SOURCE_FILES = [
    "SOURCE_00_becker_transition_models.yaml",
    "SOURCE_01_bismuth_microstructure.yaml",
    "SOURCE_02_eliminating_nongaussian_spectral_response.yaml",
]

SYNTHESIS_ID = "SYNTHESIS_01"


def find_repo_root() -> Path:
    candidates = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])
    candidates.extend(
        [
            Path("/content/sensors-becker"),
            Path("/home/dan/sensors-becker"),
            Path.home() / "sensors-becker",
        ]
    )

    for candidate in candidates:
        if (
            candidate.is_dir()
            and (candidate / "engineering_navigator").is_dir()
        ):
            return candidate

    if Path("/content").exists():
        target = Path("/content/sensors-becker")
        if not target.exists():
            subprocess.run(
                ["git", "clone", REPOSITORY_URL, str(target)],
                check=True,
            )
        return target

    raise FileNotFoundError(
        "Could not locate sensors-becker. "
        "Set REPO_ROOT_OVERRIDE to the repository path."
    )


REPO_ROOT = find_repo_root()
SOURCE_DIR = (
    REPO_ROOT
    / "engineering_navigator"
    / "absorber_manufacturing"
    / "source_records"
)
OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "engineering_questions"
    / "absorber_manufacturing"
    / SYNTHESIS_ID
)
EXPORT_DIR = REPO_ROOT / "exports" / SYNTHESIS_ID
EXPORT_ZIP = REPO_ROOT / "exports" / f"{SYNTHESIS_ID}_export.zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"Sources   : {SOURCE_DIR.relative_to(REPO_ROOT)}")
print(f"Outputs   : {OUTPUT_DIR.relative_to(REPO_ROOT)}")


Repository: /content/sensors-becker
Sources   : engineering_navigator/absorber_manufacturing/source_records
Outputs   : outputs/engineering_questions/absorber_manufacturing/SYNTHESIS_01


## 2. Load and validate the three source records

In [4]:
def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing source record: {path}")

    data = yaml.safe_load(path.read_text(encoding="utf-8"))

    if not isinstance(data, dict):
        raise TypeError(
            f"{path.name}: expected one top-level mapping"
        )

    return data


records = {}

for filename in SOURCE_FILES:
    path = SOURCE_DIR / filename
    record = load_yaml(path)

    source_id = record.get("source_id")

    if not source_id:
        raise KeyError(
            f"{filename}: missing source_id"
        )

    if source_id in records:
        raise ValueError(
            f"Duplicate source_id: {source_id}"
        )

    records[source_id] = record


if len(records) != len(SOURCE_FILES):
    raise ValueError(
        f"Expected {len(SOURCE_FILES)} source records, "
        f"loaded {len(records)}."
    )


status_rows = []

for source_id, record in records.items():
    status_rows.append(
        {
            "source_id": source_id,
            "title": record.get("title", ""),
            "record_status": record.get("record_status", ""),
            "extraction_status": record.get(
                "extraction_status",
                "",
            ),
            "reported_values": len(
                record.get("reported_values", [])
            ),
            "relationships": len(
                record.get(
                    "engineering_relationships",
                    [],
                )
            ),
        }
    )


status_df = (
    pd.DataFrame(status_rows)
    .sort_values("source_id")
    .reset_index(drop=True)
)

status_df

,source_id,title,record_status,extraction_status,reported_values,relationships
0,SOURCE_00_becker_transition_models,Use of Transition Models to Design High Perfor...,evidence_extracted,complete_for_source_record_v1,32,6
1,SOURCE_01,Microstructure analysis of bismuth absorbers f...,evidence_extracted,complete_for_source_record_v1,15,5
2,SOURCE_02,Eliminating the non-Gaussian spectral response...,evidence_extracted,complete_for_source_record_v1,26,6


In [5]:
incomplete = status_df[
    ~status_df["extraction_status"].astype(str).str.startswith("complete")
]

if not incomplete.empty:
    raise ValueError(
        "All three source records must be complete before synthesis:\n"
        + incomplete[["source_id", "extraction_status"]].to_string(index=False)
    )

print("Source-record validation: PASS")


Source-record validation: PASS


## 3. Cross-source engineering variable matrix

The records use source-specific vocabulary. This section maps recurring engineering concepts onto a shared set of synthesis axes without replacing the original source terms.


In [6]:
SYNTHESIS_AXES = {
    "critical_temperature": {
        "SOURCE_00": ["Tc"],
        "SOURCE_01": [],
        "SOURCE_02": ["Tc"],
    },
    "absorber_thickness": {
        "SOURCE_00": ["Bi_thickness"],
        "SOURCE_01": ["Bi_thickness"],
        "SOURCE_02": ["Bi_thickness"],
    },
    "deposition_method": {
        "SOURCE_00": [],
        "SOURCE_01": ["deposition_method"],
        "SOURCE_02": ["deposition_method"],
    },
    "grain_size": {
        "SOURCE_00": [],
        "SOURCE_01": ["grain_size", "SEM_grain_size", "diffraction_grain_size"],
        "SOURCE_02": ["grain_size", "average_grain_size", "average_grain_radius"],
    },
    "heat_capacity": {
        "SOURCE_00": ["C"],
        "SOURCE_01": [],
        "SOURCE_02": ["C"],
    },
    "thermal_conductance": {
        "SOURCE_00": ["G"],
        "SOURCE_01": [],
        "SOURCE_02": ["G"],
    },
    "spectral_response": {
        "SOURCE_00": ["delta_E"],
        "SOURCE_01": ["low_energy_tail"],
        "SOURCE_02": ["delta_E", "tail_fraction"],
    },
    "photon_energy": {
        "SOURCE_00": [],
        "SOURCE_01": ["secondary_electron_cloud_size"],
        "SOURCE_02": ["photon_energy"],
    },
}

variable_ids = {}
for source_id, record in records.items():
    variable_ids[source_id] = {
        item.get("id")
        for item in record.get("design_variables", [])
        if isinstance(item, dict) and item.get("id")
    }

matrix_rows = []
for axis, mapping in SYNTHESIS_AXES.items():
    row = {"engineering_axis": axis}
    support_count = 0
    for source_id in sorted(records):
        aliases = mapping.get(source_id, [])
        direct = sorted(variable_ids[source_id].intersection(aliases))
        present = bool(direct)
        row[source_id] = ", ".join(direct) if direct else "—"
        support_count += int(present)
    row["source_count"] = support_count
    matrix_rows.append(row)

variable_matrix = pd.DataFrame(matrix_rows)
variable_matrix


,engineering_axis,SOURCE_00_becker_transition_models,SOURCE_01,SOURCE_02,source_count
0,critical_temperature,—,—,—,0
1,absorber_thickness,—,Bi_thickness,Bi_thickness,2
2,deposition_method,—,deposition_method,deposition_method,2
3,grain_size,—,grain_size,grain_size,2
4,heat_capacity,—,—,C,1
5,thermal_conductance,—,—,G,1
6,spectral_response,—,low_energy_tail,"delta_E, tail_fraction",2
7,photon_energy,—,—,photon_energy,1


## 4. Quantitative evidence table

In [7]:
value_rows = []

for source_id, record in records.items():
    for item in record.get("reported_values", []):
        if not isinstance(item, dict):
            continue
        value_rows.append(
            {
                "source_id": source_id,
                "object": item.get("object"),
                "variable": item.get("variable"),
                "value": item.get("value"),
                "unit": item.get("unit"),
                "condition": item.get("condition"),
                "source_page": item.get("source_page"),
            }
        )

values_df = pd.DataFrame(value_rows)

FOCUS_VARIABLES = {
    "Tc",
    "Bi_thickness",
    "C",
    "G",
    "SEM_grain_size",
    "diffraction_grain_size",
    "average_grain_size",
    "average_grain_radius",
    "quantum_efficiency",
    "residual_resistance_ratio",
    "cloud_size",
    "delta_E",
    "predicted_delta_E",
}

focus_values = (
    values_df[values_df["variable"].isin(FOCUS_VARIABLES)]
    .sort_values(["variable", "source_id", "object"])
    .reset_index(drop=True)
)

focus_values


,source_id,object,variable,value,unit,condition,source_page
0,SOURCE_00_becker_transition_models,180 µm 3-bar device,Bi_thickness,1.1,µm,None,10
1,SOURCE_00_becker_transition_models,200 µm 4-bar device,Bi_thickness,1.1,µm,None,6
2,SOURCE_01,electroplated Bi,Bi_thickness,3.0,µm,None,2
3,SOURCE_01,evaporated Bi,Bi_thickness,3.0,µm,None,2
4,SOURCE_02,Au/Bi absorbers,Bi_thickness,3.0,µm,None,1
5,SOURCE_00_becker_transition_models,200 µm 4-bar device,C,0.166,pJ/K,None,6
6,SOURCE_00_becker_transition_models,previous device,C,0.11,pJ/K,None,5
7,SOURCE_02,large Au pixel,C,3.1,pJ/K,None,3
8,SOURCE_02,large Au/elp-Bi pixel,C,3.0,pJ/K,None,3
9,SOURCE_02,large Au/evap-Bi pixel,C,2.9,pJ/K,None,3


## 5. Recurring engineering relationships

These synthesis statements are explicit cross-source interpretations. Each statement lists the source records that contribute evidence. A statement supported by two sources is stronger than one appearing in only one source, but this notebook does not treat source count as proof.


In [8]:
synthesis_relationships = [
    {
        "id": "REL_01",
        "relationship": "Bismuth deposition method changes absorber microstructure.",
        "sources": ["SOURCE_01", "SOURCE_02"],
        "status": "supported_across_sources",
        "engineering_use": "Treat deposition method as a manufacturing specification, not merely process metadata.",
    },
    {
        "id": "REL_02",
        "relationship": "Smaller evaporated-Bi grains increase carrier scattering or trapping and are associated with low-energy spectral tailing.",
        "sources": ["SOURCE_01", "SOURCE_02"],
        "status": "supported_across_sources",
        "engineering_use": "Track grain size and morphology as absorber acceptance variables.",
    },
    {
        "id": "REL_03",
        "relationship": "Electroplated Bi preserves a Gaussian-like spectral response while adding useful x-ray stopping power with little reported heat-capacity penalty.",
        "sources": ["SOURCE_02"],
        "status": "supported_by_controlled_comparison",
        "engineering_use": "Use electroplated Bi as the leading absorber-manufacturing candidate for further process-window work.",
    },
    {
        "id": "REL_04",
        "relationship": "Absorber thickness affects detector performance through both quantum efficiency and thermalization/tailing behavior.",
        "sources": ["SOURCE_01", "SOURCE_02"],
        "status": "supported_across_sources",
        "engineering_use": "Do not optimize thickness from stopping power alone.",
    },
    {
        "id": "REL_05",
        "relationship": "TES thermal design variables C, G, Tc, geometry, and transition parameters remain coupled to absorber-manufacturing choices.",
        "sources": ["SOURCE_00", "SOURCE_02"],
        "status": "supported_across_sources",
        "engineering_use": "Evaluate absorber changes inside the full detector thermal specification.",
    },
]

relationships_df = pd.DataFrame(synthesis_relationships)
relationships_df


,id,relationship,sources,status,engineering_use
0,REL_01,Bismuth deposition method changes absorber mic...,"[SOURCE_01, SOURCE_02]",supported_across_sources,Treat deposition method as a manufacturing spe...
1,REL_02,Smaller evaporated-Bi grains increase carrier ...,"[SOURCE_01, SOURCE_02]",supported_across_sources,Track grain size and morphology as absorber ac...
2,REL_03,Electroplated Bi preserves a Gaussian-like spe...,[SOURCE_02],supported_by_controlled_comparison,Use electroplated Bi as the leading absorber-m...
3,REL_04,Absorber thickness affects detector performanc...,"[SOURCE_01, SOURCE_02]",supported_across_sources,Do not optimize thickness from stopping power ...
4,REL_05,"TES thermal design variables C, G, Tc, geometr...","[SOURCE_00, SOURCE_02]",supported_across_sources,Evaluate absorber changes inside the full dete...


## 6. Candidate leading specifications

In [9]:
candidate_specifications = [
    {
        "spec_id": "SPEC_AM_01",
        "specification": "Prefer electroplated Bi over evaporated Bi where suppression of low-energy spectral tailing is a leading detector requirement.",
        "evidence": ["SOURCE_01", "SOURCE_02"],
        "state": "source_supported_candidate",
        "next_validation": "Repeat across wafers, plating batches, absorber thicknesses, and x-ray energies.",
    },
    {
        "spec_id": "SPEC_AM_02",
        "specification": "Track absorber grain size and morphology as manufacturing acceptance variables.",
        "evidence": ["SOURCE_01", "SOURCE_02"],
        "state": "source_supported_candidate",
        "next_validation": "Determine a quantitative grain-size distribution or morphology threshold.",
    },
    {
        "spec_id": "SPEC_AM_03",
        "specification": "Treat absorber thickness as a coupled quantum-efficiency and thermalization variable.",
        "evidence": ["SOURCE_01", "SOURCE_02"],
        "state": "source_supported_candidate",
        "next_validation": "Measure tail fraction and efficiency across a controlled electroplated-Bi thickness series.",
    },
    {
        "spec_id": "SPEC_AM_04",
        "specification": "Validate absorber changes with matched thermal coupling and measured C and G.",
        "evidence": ["SOURCE_00", "SOURCE_02"],
        "state": "source_supported_candidate",
        "next_validation": "Preserve matched membrane and TES conditions across manufacturing comparisons.",
    },
]

specifications_df = pd.DataFrame(candidate_specifications)
specifications_df


,spec_id,specification,evidence,state,next_validation
0,SPEC_AM_01,Prefer electroplated Bi over evaporated Bi whe...,"[SOURCE_01, SOURCE_02]",source_supported_candidate,"Repeat across wafers, plating batches, absorbe..."
1,SPEC_AM_02,Track absorber grain size and morphology as ma...,"[SOURCE_01, SOURCE_02]",source_supported_candidate,Determine a quantitative grain-size distributi...
2,SPEC_AM_03,Treat absorber thickness as a coupled quantum-...,"[SOURCE_01, SOURCE_02]",source_supported_candidate,Measure tail fraction and efficiency across a ...
3,SPEC_AM_04,Validate absorber changes with matched thermal...,"[SOURCE_00, SOURCE_02]",source_supported_candidate,Preserve matched membrane and TES conditions a...


## 7. Unresolved specifications and next measurements

In [10]:
open_items = [
    {
        "open_specification": "Electroplated-Bi grain-size acceptance range",
        "why_open": "The sources identify grain size as important but do not define a validated tolerance.",
        "next_measurement": "Grain-size distribution versus spectral-tail fraction across replicated devices.",
    },
    {
        "open_specification": "Electroplated-Bi thickness process window",
        "why_open": "Thickness increases stopping power, while existing evidence does not establish the maximum tail-free thickness.",
        "next_measurement": "Controlled thickness series with quantum efficiency, tail fraction, C, and energy resolution.",
    },
    {
        "open_specification": "Electroplating process tolerances",
        "why_open": "Current source records do not specify allowable current-density, voltage, bath, or rate variation.",
        "next_measurement": "Process-parameter DOE linked to grain size, morphology, yield, and spectral response.",
    },
    {
        "open_specification": "Manufacturing repeatability",
        "why_open": "Wafer-to-wafer and batch-to-batch distributions are not reported in the current source set.",
        "next_measurement": "Replicated wafer/batch distributions for thickness, grain size, C, G, and detector response.",
    },
    {
        "open_specification": "Manufacturing yield",
        "why_open": "The three source records do not establish a process yield for absorber manufacturing.",
        "next_measurement": "Define pass/fail criteria and measure device yield across process batches.",
    },
]

open_specs_df = pd.DataFrame(open_items)
open_specs_df


,open_specification,why_open,next_measurement
0,Electroplated-Bi grain-size acceptance range,The sources identify grain size as important b...,Grain-size distribution versus spectral-tail f...
1,Electroplated-Bi thickness process window,"Thickness increases stopping power, while exis...",Controlled thickness series with quantum effic...
2,Electroplating process tolerances,Current source records do not specify allowabl...,"Process-parameter DOE linked to grain size, mo..."
3,Manufacturing repeatability,Wafer-to-wafer and batch-to-batch distribution...,Replicated wafer/batch distributions for thick...
4,Manufacturing yield,The three source records do not establish a pr...,Define pass/fail criteria and measure device y...


## 8. Engineering synthesis

The current source set supports a more specific absorber-manufacturing program:

```text
Electroplating process
        ↓
Bi thickness + grain-size distribution + morphology
        ↓
Carrier thermalization
        ↓
Low-energy tail fraction + energy resolution
        ↓
Manufacturing acceptance criteria
        ↓
Yield and repeatability
```

The strongest next move is therefore **not another source-extraction notebook**. It is a quantitative engineering notebook around the first unresolved specification.


In [11]:
next_notebook = {
    "id": "NB_02_ELECTROPLATED_BI_PROCESS_WINDOW",
    "engineering_question": (
        "What electroplated-Bi thickness and microstructure window preserves "
        "tail-free spectral response while increasing x-ray stopping power?"
    ),
    "inputs": [
        "SOURCE_01_bismuth_microstructure.yaml",
        "SOURCE_02_eliminating_nongaussian_spectral_response.yaml",
        "future replicated manufacturing measurements",
    ],
    "outputs": [
        "candidate thickness range",
        "candidate grain-size acceptance metric",
        "required process measurements",
        "validation experiment design",
    ],
}

next_notebook


{'id': 'NB_02_ELECTROPLATED_BI_PROCESS_WINDOW',
 'engineering_question': 'What electroplated-Bi thickness and microstructure window preserves tail-free spectral response while increasing x-ray stopping power?',
 'inputs': ['SOURCE_01_bismuth_microstructure.yaml',
  'SOURCE_02_eliminating_nongaussian_spectral_response.yaml',
  'future replicated manufacturing measurements'],
 'outputs': ['candidate thickness range',
  'candidate grain-size acceptance metric',
  'required process measurements',
  'validation experiment design']}

## 9. Write synthesis outputs and export ZIP

In [12]:
written_files = {}

status_csv = OUTPUT_DIR / "source_status.csv"
variable_matrix_csv = OUTPUT_DIR / "variable_matrix.csv"
focus_values_csv = OUTPUT_DIR / "quantitative_evidence.csv"
relationships_csv = OUTPUT_DIR / "synthesis_relationships.csv"
specifications_csv = OUTPUT_DIR / "candidate_specifications.csv"
open_specs_csv = OUTPUT_DIR / "open_specifications.csv"
synthesis_json = OUTPUT_DIR / "synthesis_summary.json"

status_df.to_csv(status_csv, index=False)
variable_matrix.to_csv(variable_matrix_csv, index=False)
focus_values.to_csv(focus_values_csv, index=False)
relationships_df.to_csv(relationships_csv, index=False)
specifications_df.to_csv(specifications_csv, index=False)
open_specs_df.to_csv(open_specs_csv, index=False)

synthesis_summary = {
    "synthesis_id": SYNTHESIS_ID,
    "sources": sorted(records),
    "candidate_specifications": candidate_specifications,
    "open_specifications": open_items,
    "next_notebook": next_notebook,
}
synthesis_json.write_text(
    json.dumps(synthesis_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

written_files = {
    "source_status": status_csv,
    "variable_matrix": variable_matrix_csv,
    "quantitative_evidence": focus_values_csv,
    "synthesis_relationships": relationships_csv,
    "candidate_specifications": specifications_csv,
    "open_specifications": open_specs_csv,
    "synthesis_summary": synthesis_json,
}

for name, path in written_files.items():
    print(f"{name:26} {path.relative_to(REPO_ROOT)}")


source_status              outputs/engineering_questions/absorber_manufacturing/SYNTHESIS_01/source_status.csv
variable_matrix            outputs/engineering_questions/absorber_manufacturing/SYNTHESIS_01/variable_matrix.csv
quantitative_evidence      outputs/engineering_questions/absorber_manufacturing/SYNTHESIS_01/quantitative_evidence.csv
synthesis_relationships    outputs/engineering_questions/absorber_manufacturing/SYNTHESIS_01/synthesis_relationships.csv
candidate_specifications   outputs/engineering_questions/absorber_manufacturing/SYNTHESIS_01/candidate_specifications.csv
open_specifications        outputs/engineering_questions/absorber_manufacturing/SYNTHESIS_01/open_specifications.csv
synthesis_summary          outputs/engineering_questions/absorber_manufacturing/SYNTHESIS_01/synthesis_summary.json


In [13]:
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for path in written_files.values():
    shutil.copy2(path, EXPORT_DIR / path.name)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(
    EXPORT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print(f"Export package: {EXPORT_ZIP}")
print(f"Size: {EXPORT_ZIP.stat().st_size:,} bytes")

try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


Export package: /content/sensors-becker/exports/SYNTHESIS_01_export.zip
Size: 5,065 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Handoff

If this synthesis runs successfully, the next engineering notebook is:

```text
NB_02_ELECTROPLATED_BI_PROCESS_WINDOW.ipynb
```

That notebook should move from source-supported relationships toward a measurable absorber-manufacturing specification.

*Admissible generalizations trail leading specifications.*
